# Role-based matched trees (human-readable)

This notebook:
- reads role definitions from `config/userconfig.json`
- loads feature-importance rows from one epsilon folder
- picks the best-matching tree for each role
- prints readable explanations and example decision paths

The repository contains eps0.005 and eps_0.010(EPSILON used as default here). For larget thresholds, run `generateSet.py`.


In [4]:
import os
import json
import ast
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')
# sklearn tree internals constants.
TREE_UNDEFINED = -2
TREE_LEAF = -1


def find_project_root(start_dir: Path) -> Path:
    for p in [start_dir] + list(start_dir.parents):
        if (p / "config" / "userconfig.json").exists() and (p / "data").exists():
            return p
    return start_dir


def load_json_flexible(path: Path):
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        raise ValueError(f"Config file is empty: {path}")
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # Handles cases like trailing commas or single-quoted content.
        return ast.literal_eval(text)


PROJECT_ROOT = find_project_root(Path.cwd())
USER_CFG_PATH = PROJECT_ROOT / "config" / "userconfig.json"





In [5]:
### CONFIG
role_features = load_json_flexible(USER_CFG_PATH)

# Same root as generateSet.py outputs
OUTPUT_ROOT = PROJECT_ROOT / "data" / "rashomonSet"
EPSILON_TO_USE = "0.010"  # change if needed, e.g. "0.050"
EPS_DIR = OUTPUT_ROOT / f"eps_{EPSILON_TO_USE}"

if not os.path.exists(EPS_DIR):
    raise FileNotFoundError(f"Epsilon folder not found: {EPS_DIR}")

FI_PATH = os.path.join(EPS_DIR, "feature_importances.csv")
if not os.path.exists(FI_PATH):
    raise FileNotFoundError(
        f"feature_importances.csv not found in {EPS_DIR}. "
        "Run analyseVariability.py first."
    )

fi = pd.read_csv(FI_PATH)


df = pd.read_csv("data/framingham_preproc.csv", index_col=0)
X = df.drop(columns=["TenYearCHD"]).fillna(df.drop(columns=["TenYearCHD"]).median())
feature_names = X.columns.tolist()


def score_tree(row, features):
    present = [f for f in features if f in row.index]
    if not present:
        return 0.0
    return float(row[present].sum())


def extract_leaf_paths(tree, feature_names):
    tree_ = tree.tree_
    left = tree_.children_left
    right = tree_.children_right
    feat = tree_.feature
    thr = tree_.threshold
    values = tree_.value

    all_paths = []

    def walk(node_id, rules, used_features):
        if left[node_id] == TREE_LEAF and right[node_id] == TREE_LEAF:
            counts = values[node_id][0]
            total = counts.sum()
            prob = float(counts[1] / total) if total > 0 else 0.0
            all_paths.append(
                {
                    "risk": prob,
                    "rules": list(rules),
                    "used_features": set(used_features),
                }
            )
            return

        feat_name = feature_names[feat[node_id]]
        threshold = thr[node_id]
        next_features = used_features | {feat_name}

        walk(
            left[node_id],
            rules + [f"{feat_name} is <= {threshold:.2f}"],
            next_features,
        )
        walk(
            right[node_id],
            rules + [f"{feat_name} is > {threshold:.2f}"],
            next_features,
        )

    walk(0, [], set())
    return all_paths


def pick_low_high_examples(tree, feature_names, role_feats):
    paths = extract_leaf_paths(tree, feature_names)
    if not paths:
        return None, None

    role_set = set(role_feats)
    tree_used_features = {
        feature_names[idx]
        for idx in tree.tree_.feature
        if idx >= 0
    }
    tree_role_features = tree_used_features & role_set

    for p in paths:
        p["role_overlap"] = len(p["used_features"] & role_set)
        p["path_feature_count"] = len(p["used_features"])

    if tree_role_features:
        # Tree uses role features: prefer paths that include the most of them.
        max_overlap = max(p["role_overlap"] for p in paths)
        candidate_paths = [p for p in paths if p["role_overlap"] == max_overlap and max_overlap > 0]
        if not candidate_paths:
            candidate_paths = paths
    else:
        # Tree has no role features: use the richest (most informative) paths.
        max_feature_count = max(p["path_feature_count"] for p in paths)
        candidate_paths = [p for p in paths if p["path_feature_count"] == max_feature_count]

    low_example = min(candidate_paths, key=lambda p: p["risk"])
    high_example = max(candidate_paths, key=lambda p: p["risk"])
    return low_example, high_example


def risk_band(prob):
    if prob < 0.20:
        return "low"
    return "high"


In [7]:
# User-wise matching and explanation

for role, feats in role_features.items():
    print("\n" + "=" * 68)
    print(f"User: {role}")
    print("=" * 68)
    print("User aligned features:", ", ".join(feats))

    role_scores = fi.apply(lambda row: score_tree(row, feats), axis=1)
    best_tree_idx = int(role_scores.idxmax())
    best_score = float(role_scores.iloc[best_tree_idx])

    tree_path = os.path.join(EPS_DIR, f"tree_{best_tree_idx}.pkl")
    with open(tree_path, "rb") as f:
        tree = pickle.load(f)

    print(f"Matched tree: tree_{best_tree_idx}.pkl")
    print(f"Match score (sum of user-feature importances): {best_score:.4f}")

    row = fi.loc[best_tree_idx]
    ranked = row[feature_names].sort_values(ascending=False)

    print("\nFeatures in descending order of importance in this tree:")
    for feat, val in ranked.items():
        print(f"- {feat}: {val:.4f}")

    print("\nUser features in this tree:")
    for feat in feats:
        if feat in ranked.index:
            print(f"- {feat}: {ranked[feat]:.4f}")
        else:
            print(f"- {feat}: not present in model features")

    low_example, high_example = pick_low_high_examples(tree, feature_names, feats)

    print("\nExample decisions from this tree (leaf paths):")

    if low_example is not None:
        prob = low_example["risk"]
        print(f"\nLow-risk example: predicted risk = {prob:.3f} ({risk_band(prob)})")
        print("Decision path:")
        for rule in low_example["rules"]:
            print(f"  - {rule}")

    if high_example is not None:
        prob = high_example["risk"]
        print(f"\nHigh-risk example: predicted risk = {prob:.3f} ({risk_band(prob)})")
        print("Decision path:")
        for rule in high_example["rules"]:
            print(f"  - {rule}")



User: doctor
User aligned features: MAP, totChol, diabetes_grade
Matched tree: tree_8.pkl
Match score (sum of user-feature importances): 0.3378

Features in descending order of importance in this tree:
- age: 0.2993
- MAP: 0.2577
- male: 0.0962
- prevalentHyp: 0.0945
- totChol: 0.0801
- BMI: 0.0727
- heartRate: 0.0339
- cigsPerDay: 0.0311
- diabetes: 0.0263
- currentSmoker: 0.0082
- BPMeds: 0.0000
- prevalentStroke: 0.0000
- diabetes_grade: 0.0000

User features in this tree:
- MAP: 0.2577
- totChol: 0.0801
- diabetes_grade: 0.0000

Example decisions from this tree (leaf paths):

Low-risk example: predicted risk = 0.032 (low)
Decision path:
  - age is <= 48.50
  - MAP is > 104.58
  - totChol is <= 258.50
  - cigsPerDay is <= 14.50
  - BMI is <= 30.15

High-risk example: predicted risk = 0.536 (high)
Decision path:
  - age is <= 48.50
  - MAP is > 104.58
  - totChol is > 258.50
  - male is > 0.50
  - heartRate is > 79.00

User: patient
User aligned features: cigsPerDay, BMI
Matched tre